# Laboratorio L4 - Riassumere i dati

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessari nella stessa cartella: `pinguini.csv`, `tragitti_esempio.csv`.

Dati: Palmer Penguins, Horst AM, Hill AP, Gorman KB (2020), licenza CC0, https://allisonhorst.github.io/palmerpenguins/

In [1]:
import pandas as pd

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

pinguini = pd.read_csv("pinguini.csv")

## 1. Statistiche di una variabile

- `mean()` media, `median()` mediana, `std()` deviazione standard, `min()`, `max()`
- `quantile(0.25)` primo quartile, `quantile(0.75)` terzo quartile
- i valori mancanti vengono ignorati automaticamente

In [2]:
massa = pinguini["massa_g"]
print("media     :", round(massa.mean(), 1))
print("mediana   :", massa.median())
print("dev. std. :", round(massa.std(), 1))
print("minimo    :", massa.min(), " massimo:", massa.max())
print("Q1, Q3    :", massa.quantile(0.25), massa.quantile(0.75))

media     : 4201.8
mediana   : 4050.0
dev. std. : 802.0
minimo    : 2700.0  massimo: 6300.0
Q1, Q3    : 3550.0 4750.0


`describe()` calcola tutte queste statistiche per ogni colonna numerica: `count` è il numero di valori presenti, `25%`, `50%`, `75%` sono i quartili (il 50% è la mediana).

In [3]:
pinguini.describe().round(1)

,becco_lunghezza_mm,becco_profondita_mm,pinna_lunghezza_mm,massa_g,anno
count,342.0,342.0,342.0,342.0,344.0
mean,43.9,17.2,200.9,4201.8,2008.0
std,5.5,2.0,14.1,802.0,0.8
min,32.1,13.1,172.0,2700.0,2007.0
25%,39.2,15.6,190.0,3550.0,2007.0
50%,44.4,17.3,197.0,4050.0,2008.0
75%,48.5,18.7,213.0,4750.0,2009.0
max,59.6,21.5,231.0,6300.0,2009.0


Completare: calcolare la mediana della lunghezza della pinna.

In [4]:
mediana_pinna = pinguini["pinna_lunghezza_mm"].median()
print(mediana_pinna)
controlla(mediana_pinna == 197.0, "Il metodo che calcola la mediana è median().")

197.0
Corretto.


## 2. Statistiche per gruppo (esercizio 1)

- `groupby("specie")`: divide le righe in gruppi, uno per specie
- `["massa_g"].mean()`: calcola la media della massa in ciascun gruppo
- `agg([...])`: calcola più statistiche insieme

In [5]:
pinguini.groupby("specie")["massa_g"].mean().round(0)

specie
Adelie       3701.0
Chinstrap    3733.0
Gentoo       5076.0
Name: massa_g, dtype: float64

In [6]:
misure = ["becco_lunghezza_mm", "becco_profondita_mm", "pinna_lunghezza_mm", "massa_g"]
pinguini.groupby("specie")[misure].mean().round(1)

,becco_lunghezza_mm,becco_profondita_mm,pinna_lunghezza_mm,massa_g
specie,,,,
Adelie,38.8,18.3,190.0,3700.7
Chinstrap,48.8,18.4,195.8,3733.1
Gentoo,47.5,15.0,217.2,5076.0


In [7]:
pinguini.groupby("specie")["massa_g"].agg(["count", "mean", "median", "std", "min", "max"]).round(0)

,count,mean,median,std,min,max
specie,,,,,,
Adelie,151,3701.0,3700.0,459.0,2850.0,4775.0
Chinstrap,68,3733.0,3700.0,384.0,2700.0,4800.0
Gentoo,123,5076.0,5000.0,504.0,3950.0,6300.0


**Domanda**: osservando le medie per specie, quale misura distingue meglio i Gentoo dalle altre due specie? Quale misura distingue meglio Adelie da Chinstrap?

Risposta:

## 3. Due variabili di raggruppamento (esercizio 2)

`groupby(["specie", "sesso"])` crea un gruppo per ogni combinazione di specie e sesso.

In [8]:
pinguini.groupby(["specie", "sesso"])["massa_g"].mean().round(0)

specie     sesso  
Adelie     femmina    3369.0
           maschio    4043.0
Chinstrap  femmina    3527.0
           maschio    3939.0
Gentoo     femmina    4680.0
           maschio    5485.0
Name: massa_g, dtype: float64

Completare: calcolare, per specie e sesso, la media della lunghezza del becco.

In [9]:
becco = pinguini.groupby(["specie", "sesso"])["becco_lunghezza_mm"].mean().round(1)
print(becco)
controlla(becco[("Gentoo", "maschio")] == 49.5, "La colonna è \"becco_lunghezza_mm\".")

specie     sesso  
Adelie     femmina    37.3
           maschio    40.4
Chinstrap  femmina    46.6
           maschio    51.1
Gentoo     femmina    45.6
           maschio    49.5
Name: becco_lunghezza_mm, dtype: float64
Corretto.


## 4. Tabelle di frequenza (esercizio 2)

- `pd.crosstab(righe, colonne)`: tabella a doppia entrata con i conteggi di ogni combinazione
- `normalize="index"`: percentuali per riga invece dei conteggi

In [10]:
pd.crosstab(pinguini["isola"], pinguini["specie"])

specie,Adelie,Chinstrap,Gentoo
isola,,,
Biscoe,44,0,124
Dream,56,68,0
Torgersen,52,0,0


In [11]:
(pd.crosstab(pinguini["isola"], pinguini["specie"], normalize="index") * 100).round(1)

specie,Adelie,Chinstrap,Gentoo
isola,,,
Biscoe,26.2,0.0,73.8
Dream,45.2,54.8,0.0
Torgersen,100.0,0.0,0.0


## 5. Ordinare (esercizio 3)

`sort_values("colonna", ascending=False)` ordina le righe dalla più grande alla più piccola.

In [12]:
pinguini.sort_values("massa_g", ascending=False).head(5)

,specie,isola,becco_lunghezza_mm,becco_profondita_mm,pinna_lunghezza_mm,massa_g,sesso,anno
169,Gentoo,Biscoe,49.2,15.2,221.0,6300.0,maschio,2007
185,Gentoo,Biscoe,59.6,17.0,230.0,6050.0,maschio,2007
269,Gentoo,Biscoe,48.8,16.2,222.0,6000.0,maschio,2009
229,Gentoo,Biscoe,51.1,16.3,220.0,6000.0,maschio,2008
231,Gentoo,Biscoe,45.2,16.4,223.0,5950.0,maschio,2008


## 6. Media e mediana del tempo di tragitto (esercizio 4)

Il file dei tragitti ha la colonna `tempo_min` letta come testo (L3). Per ora si considerano solo le risposte scritte correttamente come numero: `pd.to_numeric(..., errors="coerce")` converte in numero e trasforma in `NaN` ciò che non riesce a convertire. La pulizia completa è l'oggetto di L5.

In [13]:
tragitti = pd.read_csv("tragitti_esempio.csv")
tempo = pd.to_numeric(tragitti["tempo_min"], errors="coerce")
print("risposte convertite:", tempo.count(), "su", len(tempo))
print("media  :", round(tempo.mean(), 1))
print("mediana:", tempo.median())

risposte convertite: 139 su 153
media  : 30.0
mediana: 23.0


**Domande**

- La media è più grande della mediana. Quali valori la "tirano" verso l'alto? Usare la cella seguente per vedere i tempi più grandi.
- Quale delle due misure descrive meglio il tempo di un tragitto "tipico"?

Risposte:

In [14]:
tempo.sort_values(ascending=False).head(8)

75     300.0
116     75.0
130     68.0
87      64.0
33      63.0
120     60.0
68      60.0
151     60.0
Name: tempo_min, dtype: float64

## 7. Approfondimento (esercizio 5)

- `std()` per gruppo: quale specie ha la massa più variabile?
- scarto interquartile (IQR) = Q3 - Q1

Completare: calcolare lo scarto interquartile della massa dei Gentoo.

In [15]:
gentoo = pinguini[pinguini["specie"] == "Gentoo"]["massa_g"]
iqr = gentoo.quantile(0.75) - gentoo.quantile(0.25)
print(iqr)
controlla(iqr == 800.0)

800.0
Corretto.


In [16]:
pinguini.groupby("specie")["massa_g"].std().round(0)

specie
Adelie       459.0
Chinstrap    384.0
Gentoo       504.0
Name: massa_g, dtype: float64